# Experiment 02: Linear Recovery

Check whether the linear baseline recovers the discrete-time sinusoid structure under easy conditions.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


Repo root: C:\Users\WWindows10\Documents\github_project\python-neural-network-research


In [2]:
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

from src import ExperimentConfig, run_experiment


def overall_row(results: dict) -> pd.Series:
    summary_df = results["summary_df"]
    return summary_df.loc[summary_df["set_idx"] == "overall"].iloc[0]


def collect_overall_rows(result_map: dict[str, dict], columns: list[str] | None = None) -> pd.DataFrame:
    rows = []
    for run_label, result in result_map.items():
        row = overall_row(result).copy()
        row["run_label"] = run_label
        rows.append(row)
    df = pd.DataFrame(rows)
    if columns is not None:
        return df[["run_label", *columns]]
    return df


In [3]:
BASE_CONFIG = dict(
    time_mode="discrete",
    SEQ_LEN=4096,
    theta_min=0.05 * np.pi,
    theta_max=0.85 * np.pi,
    MIN_DELTA_THETA=0.04 * np.pi,
    RANDOM_AMPLITUDE=False,
    RANDOM_PHASE=True,
    USE_NOISE=False,
    HIDDEN_DIM=128,
    BOTTLENECK_MULTIPLIER=4,
    NUM_EXPERIMENTS=20,
    SEEDS_PER_FREQ=5,
    VAL_RATIO=0.2,
    TEST_RATIO=0.2,
    NORMALIZE_H_COLUMNS=False,
    VERBOSE=False,
    MAKE_PLOTS=False,
)


In [4]:
def run_linear_recovery(num_freqs: int, lag: int) -> dict:
    cfg = ExperimentConfig(
        MODEL_ID="AN003_LINEAR",
        NUM_FREQS=num_freqs,
        LAG=lag,
        LR=1e-2,
        EPOCHS=1500,
        **BASE_CONFIG,
    )
    return run_experiment(cfg)


In [5]:
results = {}
condition_specs = [(num_freqs, lag_multiplier * num_freqs) for num_freqs in [2, 4, 8] for lag_multiplier in [2, 4, 8]]
for num_freqs, lag in tqdm(condition_specs, desc="experiment 2 conditions"):
    label = f"k={num_freqs},L={lag}"
    results[label] = run_linear_recovery(num_freqs=num_freqs, lag=lag)

collect_overall_rows(
    results,
    columns=[
        "test_mse_mean",
        "test_r2_mean",
        "test_align_coverage_full_mean",
        "test_recon_r2_qf_from_h_mean",
        "test_align_mean_angle_deg_full_mean",
        "train_rank_threshold_mean",
        "test_rank_threshold_mean",
        "train_rank_entropy_mean",
        "test_rank_entropy_mean",
        "train_h_numerical_dim_mean",
        "test_h_numerical_dim_mean",
    ],
)


C:\Users\WWindows10\Documents\github_project\python-neural-network-research\src\experiment_runner.py:433: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary_df, pd.DataFrame([overall_row])], ignore_index=True)
C:\Users\WWindows10\Documents\github_project\python-neural-network-research\src\experiment_runner.py:433: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary_df, pd.DataFrame([overall_row])], ignore_index=True)
C:\Users\WWindows10\Documents\github_p

,run_label,test_mse_mean,test_r2_mean,test_align_coverage_full_mean,test_recon_r2_qf_from_h_mean,test_align_mean_angle_deg_full_mean
20,"k=2,L=4",1.343780e-04,0.999866,1.0,1.0,0.000024
20,"k=2,L=8",1.189659e-06,0.999999,1.0,1.0,0.000012
20,"k=2,L=16",2.997248e-07,1.000000,1.0,1.0,0.000011
20,"k=4,L=8",2.862548e-03,0.998571,1.0,1.0,0.000397
20,"k=4,L=16",3.256216e-05,0.999984,1.0,1.0,0.000035
20,"k=4,L=32",1.777655e-03,0.999119,1.0,1.0,0.000026
20,"k=8,L=16",4.634302e-02,0.988413,1.0,1.0,0.002100
20,"k=8,L=32",1.873139e-02,0.995343,1.0,1.0,0.000033
20,"k=8,L=64",1.614862e-02,0.995961,1.0,1.0,0.000038


In [ ]:
def collect_all_scalar_overall_metrics(result_map: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for run_label, result in result_map.items():
        overall = result["overall_summary"]
        row = {"run_label": run_label}
        for key, value in overall.items():
            if np.isscalar(value):
                row[key] = value
        rows.append(row)
    return pd.DataFrame(rows).sort_values("run_label").reset_index(drop=True)

all_overall_metrics_df = collect_all_scalar_overall_metrics(results)
all_overall_metrics_df
